In [2]:
import importlib
import random

from deap import base, creator, tools

import params.params as params
import params.seeding as seeding


def refresh_configuration():
    """Reload configuration modules so notebook cells see source changes."""
    global params, seeding
    params = importlib.reload(params)
    seeding = importlib.reload(seeding)
    return params, seeding

In [3]:
# Define the fitness to maximize
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Define the Individual class as a list of Gene objects with an associated fitness.
creator.create("Individual", list, fitness=creator.FitnessMax)

In [4]:
def create_gene(assigned_node_id):
    """Generates a single active sensor configuration."""
    sensor_type = random.choice(list(params.SensorType))

    # Define physical rotation bounds.
    # Note: If you are using 360-degree spinning LiDARs, roll/pitch is sufficient.
    # If using directional solid-state LiDARs, you may need to swap 'roll' for 'yaw'.
    pitch = random.randint(-90, 90)
    roll = random.randint(-180, 180)

    return params.Gene(sensor_type=sensor_type, node_id=assigned_node_id, pitch=pitch, roll=roll)


def create_individual(icls):
    """Generates an individual with 1 to 4 active sensors."""
    num_sensors = random.randint(1, params.MAX_SENSORS_PER_INDIVIDUAL)

    # Ensure unique node IDs for each sensor in the individual.
    unique_nodes = random.sample(params.VALID_NODE_IDS, num_sensors)

    genes = [create_gene(node_id) for node_id in unique_nodes]

    # Initialize the DEAP list class with our generated genes.
    return icls(genes)


def create_seeded_population(icls, individual_factory, seed_contents, population_size, shuffle=False):
    """Builds a population from seeded individuals plus random fill."""
    seeded_population = [icls(seed) for seed in seed_contents]

    if len(seeded_population) > population_size:
        raise ValueError(
            f"seed_contents contains {len(seeded_population)} individuals, "
            f"but population_size is only {population_size}."
        )

    random_population = [individual_factory() for _ in range(population_size - len(seeded_population))]
    population = seeded_population + random_population

    if shuffle:
        random.shuffle(population)

    return population

In [30]:
toolbox = base.Toolbox()

# Register the individual generator.
toolbox.register("individual", create_individual, creator.Individual)

# Register the population generator (creates a list of individuals).
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# NOTE: Consider Demes
# A deme is a sub-population that is contained in a population.
# https://deap.readthedocs.io/en/master/tutorials/basic/part1.html#demes

# Create initial population.
refresh_configuration()
population = create_seeded_population(
    creator.Individual,
    toolbox.individual,
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

In [ ]:
import importlib

from utils import visualization as visualization_module

visualization = importlib.reload(visualization_module)
visualization.visualize_population(population, max_display=10)

Individual,Sensor 1,Sensor 2,Sensor 3
Individual 1,"LIDAR_16_CHnode 5pitch 0, roll 0",—,—
Individual 2,"SOLID_STATEnode 42pitch 10, roll -15","LIDAR_32_CHnode 99pitch -5, roll 20",—
Individual 3,"SOLID_STATEnode 180pitch 42, roll 6",—,—
Individual 4,"SOLID_STATEnode 67pitch -1, roll -88","LIDAR_16_CHnode 123pitch -15, roll 125","LIDAR_32_CHnode 138pitch -31, roll 136"
Individual 5,"LIDAR_32_CHnode 37pitch -8, roll -162",—,—
